## Poke the LLM

In [ ]:
import os, json
os.chdir(os.path.abspath("../triage"))
from triage.llm import call, text_of, client, LLMError

### 01. Anatomy of a response

In [ ]:
r = call("In one sentence: what is a connection pool?", tag="e1")
print(r.model_dump_json(indent=2))

```code
{
  "id": "msg_011CgmfziC1ZSarZr7fjUUN9",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "A connection pool is a cache of reusable, pre-established database (or other resource) connections that applications borrow and return instead of opening and closing a new connection for every request, which saves time and limits resource use.",
      "type": "text"
    }
  ],
  "diagnostics": null,
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
...
    "service_tier": "standard",
    "speed": "standard"
  }
}
Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

In [ ]:
import json

def pretty(r):
    d = r.model_dump(exclude_none=True)                  # drop null fields
    for b in d["content"]:
        if "signature" in b:                              # shorten the long signature
            b["signature"] = b["signature"][:24] + f"... ({len(b['signature'])} chars)"
    print(json.dumps(d, indent=2))

r = call("In one sentence: what is a connection pool?", tag="e1")
pretty(r)

```code
{
  "id": "msg_011CgmgPvPPR6K5xLoudYwAv",
  "content": [
    {
      "text": "A connection pool is a cache of reusable, pre-established connections (such as database connections) that applications borrow and return instead of opening and closing a new connection for every request, which saves time and resources.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 22,
    "output_tokens": 61,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

### 02. Truncation

In [ ]:
r = client.messages.create(
        model="claude-haiku-5-5",
        max_tokens=30,
        messages=[{"role": "user", "content": "List 20 causes of HTTP 5xx errors."}])
print(r.stop_reason, [b.type for b in r.content], r.usage.output_tokens)
print(repr(text_of(r)))

try:
    call("List 20 causes of HTTP 5xx errors.", max_tokens=30, tag="e2")
except LLMError as e:
    print("wrapper raised:", e)

```code
max_tokens ['text'] 30
'Here are 20 common causes of HTTP 5xx errors:\n\n**Server-side software issues'
wrapper raised: truncated at max_tokens=30 (a7e768ca70ff)

In [ ]:
pretty(r)

```code
{
  "id": "msg_011Cgmh3Mfge4YBFkktpQNuD",
  "content": [
    {
      "text": "Here are 20 common causes of HTTP 5xx errors:\n\n**Server-side software issues",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "max_tokens",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 26,
    "output_tokens": 30,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

### 03. Effort: Low vs High

In [ ]:
q = ("Diagnose in 3 bullets: p99 latency doubled right after a deploy "
     "that changed DB_POOL_SIZE from 50 to 5.")

for eff in ("low", "high"):
    r = call(q, effort=eff, tag=f"e3-{eff}")
    print(f"--- {eff}\n{text_of(r)}")
    print()
    pretty(r)

for line in open("runs.jsonl").readlines()[-2:]:
    d = json.loads(line)
    print(d["effort"], d["output_tokens"], d["thinking_tokens"], d["cost_usd"], d["latency_s"])

```code
--- low
Likely causes, in order of probability:

- **Connection starvation (most likely):** Dropping the pool from 50 to 5 means requests now queue waiting for a free connection. Check connection-acquire wait time or pool wait metrics (e.g., HikariCP `pool.wait`, SQLAlchemy `pool_timeout` waits). If those spiked, the added latency is queueing, not slower queries. Under any concurrency above ~5 in-flight DB operations, requests stack up behind the pool.

- **Pool too small for peak concurrency:** Compare DB_POOL_SIZE against your actual concurrent DB demand (requests/sec × average DB time per request, by Little's Law: connections needed ≈ throughput × DB hold time). If that product exceeds 5 at peak, p99 will be dominated by wait time, which is the tail that doubles first. Also verify that the old value of 50 wasn't masking a real problem, and that the DB itself can handle the load you're now funneling through fewer connections.

- **Confirm it's the pool and not something else in the deploy:** Diff the rest of the release (other config, code paths, dependency bumps, or a new slow query), and check whether the latency increase tracks pool wait time or DB query time. If DB query duration is flat and pool wait rose, the pool change is the cause. Quick mitigation: roll back or raise the pool to a value sized from measured concurrency (e.g., 2–3× peak in-flight DB calls per instance, scaled by total instance count against the DB's max_connections).

{
  "id": "msg_011CgmhKioNvzwJXFMwZWhaY",
  "content": [
    {
      "text": "Likely causes, in order of probability:\n\n- **Connection starvation (most likely):** Dropping the pool from 50 to 5 means requests now queue waiting for a free connection. Check connection-acquire wait time or pool wait metrics (e.g., HikariCP `pool.wait`, SQLAlchemy `pool_timeout` waits). If those spiked, the added latency is queueing, not slower queries. Under any concurrency above ~5 in-flight DB operations, requests stack up behind the pool.\n\n- **Pool too small for peak concurrency:** Compare DB_POOL_SIZE against your actual concurrent DB demand (requests/sec \u00d7 average DB time per request, by Little's Law: connections needed \u2248 throughput \u00d7 DB hold time). If that product exceeds 5 at peak, p99 will be dominated by wait time, which is the tail that doubles first. Also verify that the old value of 50 wasn't masking a real problem, and that the DB itself can handle the load you're now funneling through fewer connections.\n\n- **Confirm it's the pool and not something else in the deploy:** Diff the rest of the release (other config, code paths, dependency bumps, or a new slow query), and check whether the latency increase tracks pool wait time or DB query time. If DB query duration is flat and pool wait rose, the pool change is the cause. Quick mitigation: roll back or raise the pool to a value sized from measured concurrency (e.g., 2\u20133\u00d7 peak in-flight DB calls per instance, scaled by total instance count against the DB's max_connections).",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 57,
    "output_tokens": 510,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}
--- high
- **Most likely cause:** The pool shrank 10x (50 → 5), so requests now queue for a free connection. Any request that waits for checkout adds its wait time directly to latency, and p99 captures that queueing first.

- **Why p99 doubled, not just the average:** Queueing is nonlinear. At moderate load, a few requests hit a busy pool and wait for a connection held by a slow query, and those waits dominate the tail. Average latency may look nearly unchanged while p99 jumps.

- **How to confirm and fix:** Check pool wait or checkout time metrics and pool timeout errors. If waits are high, the 5-connection pool is too small for your concurrency. Revert to 50, or size it from load (roughly concurrency × average query time) and confirm the total across all app instances and workers stays under the DB's `max_connections`.

{
  "id": "msg_011CgmhL1yjzaSKs5GEFY3YZ",
  "content": [
    {
      "signature": "CAQS8gcKEAgTGAI4AUIIdGhp... (1356 chars)",
      "thinking": "",
      "type": "thinking"
    },
    {
      "text": "- **Most likely cause:** The pool shrank 10x (50 \u2192 5), so requests now queue for a free connection. Any request that waits for checkout adds its wait time directly to latency, and p99 captures that queueing first.\n\n- **Why p99 doubled, not just the average:** Queueing is nonlinear. At moderate load, a few requests hit a busy pool and wait for a connection held by a slow query, and those waits dominate the tail. Average latency may look nearly unchanged while p99 jumps.\n\n- **How to confirm and fix:** Check pool wait or checkout time metrics and pool timeout errors. If waits are high, the 5-connection pool is too small for your concurrency. Revert to 50, or size it from load (roughly concurrency \u00d7 average query time) and confirm the total across all app instances and workers stays under the DB's `max_connections`.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 57,
    "output_tokens": 429,
    "output_tokens_details": {
      "thinking_tokens": 150
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}
low 510 0 0.000261 4.067
high 429 150 0.00022 3.064

### 04. count_tokens vs actual

In [ ]:
msgs = [{"role": "user", "content": q}]
est = client.messages.count_tokens(model="claude-haiku-5-5", messages=msgs).input_tokens
r = call(msgs, tag="e4")
print("estimate:", est, "actual:", r.usage.input_tokens)
print()
pretty(r)

```code
estimate: 57 actual: 57

{
  "id": "msg_011CgmhmwhEQTsdyCHeE8rQB",
  "content": [
    {
      "text": "- **Connection starvation (most likely):** Dropping the pool from 50 to 5 means requests queue waiting for a free connection. Under normal load, many requests now wait on checkout, which shows up as a tail-latency spike (p99 hits hardest, since the slowest waiters are the ones stuck in line). Check pool wait time or checkout timeout metrics; if your ORM or driver exposes them (e.g., HikariCP `pool.wait`, SQLAlchemy `pool_timeout` waits), a jump there confirms it.\n\n- **Concurrency now exceeds capacity:** If each app instance handles more concurrent requests than 5 connections can serve, throughput drops and queueing compounds. Compare in-flight requests per instance and DB active connections before vs. after. Also check whether the pool is per-process: with N instances and workers, the total could have shrunk far more than intended, or the setting may have been applied to only some pods, creating uneven latency.\n\n- **Quick verification and fix:** Roll back or temporarily set DB_POOL_SIZE back to 50 in one instance and compare its p99 against the others. If that instance recovers, the pool size is the cause. Then size the pool from data: roughly (peak concurrent DB-using requests per instance) with headroom, while keeping total connections across all instances under the DB's max_connections. If 5 was intended to reduce DB load, the tradeoff is queueing latency, so consider a pool size that balances both or fixing the slow queries holding connections longer.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 57,
    "output_tokens": 497,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

### 05. multi-turn state

In [ ]:
msgs = [{"role": "user", "content": "Pick a random HTTP status code. Reply with only the number."}]
r1 = call(msgs, tag="e5")
msgs += [{"role": "assistant", "content": r1.content},     # full blocks, thinking included
         {"role": "user", "content": "What does it mean? One line."}]

pretty(r1)
print()

r2 = call(msgs, tag="e5")
print(text_of(r1), "->", text_of(r2))
print("turn-2 input tokens:", r2.usage.input_tokens, "vs turn-1:", r1.usage.input_tokens)

pretty(r2)
print()

r3 = call("What does it mean? One line.", tag="e5-nohistory")
print("no history:", text_of(r3))
pretty(r3)
print()

```code
{
  "id": "msg_011CgmhvN79TVgwboJSbnBqQ",
  "content": [
    {
      "text": "418",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 28,
    "output_tokens": 3,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

418 -> I'm a teapot: the server refuses to brew coffee because it is a teapot (an April Fools' joke from RFC 2324, and a playful error response).
turn-2 input tokens: 42 vs turn-1: 28
{
  "id": "msg_011CgmhvVNx3GAtcWojmhkQZ",
  "content": [
    {
      "text": "I'm a teapot: the server refuses to brew coffee because it is a teapot (an April Fools' joke from RFC 2324, and a playful error response).",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 42,
    "output_tokens": 52,
    "output_tokens_details": {
      "thinking_tokens": 0
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

no history: I don't see what "it" refers to. Could you share the word, phrase, or text you're asking about?
{
  "id": "msg_011CgmhvdHDQQuCumwG3PMDx",
  "content": [
    {
      "signature": "CAQS2gUKEAgTGAI4AUIIdGhp... (984 chars)",
      "thinking": "",
      "type": "thinking"
    },
    {
      "text": "I don't see what \"it\" refers to. Could you share the word, phrase, or text you're asking about?",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 17,
    "output_tokens": 76,
    "output_tokens_details": {
      "thinking_tokens": 43
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}


### 06. seeing the thinking

In [ ]:
r = client.messages.create(model="claude-haiku-5-5", max_tokens=4000,
        thinking={"type": "adaptive", "display": "summarized"},
        output_config={"effort": "high"},
        messages=[{"role": "user", "content": q}])
for b in r.content:
    print(b.type, "->", (b.thinking if b.type == "thinking" else b.text)[:400], "\n")

```
thinking -> I'm weighing a three-bullet diagnosis: the pool shrink from 50 to 5 is causing connection contention, so requests queue for a free connection, which hits p99 hardest. I'm thinking about how the bullets should cover the likely cause, how to confirm it, and the fix.

I'm laying out the confirmation step: checking pool checkout wait times and utilization near 100%, and whether the database side stays 

text -> - **Likely cause:** Pool checkout queuing. Dropping from 50 to 5 connections means requests wait for a free connection whenever more than 5 are in flight. Waits are small on average but pile up in the tail, so p99 moves much more than p50.
- **Confirm:** Check pool acquire-wait time and pool utilization (near 100% at peak). If p50 barely moved while p99 doubled, and DB query durations are unchange 


In [ ]:
pretty(r)

```code
{
  "id": "msg_011CgmhzooqBa5oRFN2JcivB",
  "content": [
    {
      "signature": "CAQS6Q0KEAgTGAI4AUIIdGhp... (2368 chars)",
      "thinking": "I'm weighing a three-bullet diagnosis: the pool shrink from 50 to 5 is causing connection contention, so requests queue for a free connection, which hits p99 hardest. I'm thinking about how the bullets should cover the likely cause, how to confirm it, and the fix.\n\nI'm laying out the confirmation step: checking pool checkout wait times and utilization near 100%, and whether the database side stays flat. A p99 that doubles while p50 barely moves would point to queueing rather than slower queries. For the fix, I'm weighing resizing the pool against database capacity, and making sure the deploy didn't change timeouts or leak connections.\n\n",
      "type": "thinking"
    },
    {
      "text": "- **Likely cause:** Pool checkout queuing. Dropping from 50 to 5 connections means requests wait for a free connection whenever more than 5 are in flight. Waits are small on average but pile up in the tail, so p99 moves much more than p50.\n- **Confirm:** Check pool acquire-wait time and pool utilization (near 100% at peak). If p50 barely moved while p99 doubled, and DB query durations are unchanged, the extra latency is queueing, not slower queries.\n- **Fix:** Revert to 50 as a quick mitigation, then size the pool from measured concurrency rather than a guess. A starting estimate is peak concurrent requests \u00d7 fraction that touch the DB, bounded by the DB's `max_connections` headroom.",
      "type": "text"
    }
  ],
  "model": "claude-haiku-5-5",
  "role": "assistant",
  "stop_reason": "end_turn",
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "global",
    "input_tokens": 57,
    "output_tokens": 671,
    "output_tokens_details": {
      "thinking_tokens": 419
    },
    "service_tier": "standard",
    "speed": "standard"
  }
}

#### END